# Focus settoriale

            Questo foglio analizza come la distribuzione del valore aggiunto per classe dimensionale cambia tra manifattura, costruzioni, commercio e servizi di mercato inclusi nella configurazione.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from valore_aggiunto_imprese.analysis import (
    METRIC_LABELS,
    add_source_note,
    latest_common_year_with_values,
    latest_year_with_values,
    SIZE_ORDER_BUSINESS,
    SIZE_ORDER_OFFICIAL,
    add_share,
    enrich_business_demography,
    enrich_sbs,
    read_project_csv,
)

pd.options.display.max_columns = 80
pd.options.display.float_format = "{:,.2f}".format

## Dataset settoriale

In [ ]:
sbs = enrich_sbs(read_project_csv("data/processed/official_eurostat_sbs.csv"))
latest_year = latest_year_with_values(sbs, metric_code="AV_MEUR")
sector_va = sbs.query("metric_code == 'AV_MEUR' and year == @latest_year").dropna(
    subset=["value"]
)
sector_va[["country_label", "sector_label", "size_label", "value"]].head()

## Valore aggiunto per settore

In [ ]:
sector_totals = (
    sector_va.groupby(["country_label", "sector_label"], observed=True, as_index=False)["value"]
    .sum()
)

fig = px.bar(
    sector_totals,
    x="country_label",
    y="value",
    color="sector_label",
    labels={"country_label": "Paese", "value": "Milioni di euro", "sector_label": "Settore"},
    title=f"Valore aggiunto per settore - {latest_year}",
)
add_source_note(fig, "Eurostat Structural Business Statistics")
fig

## Italia: composizione settoriale per classe

In [ ]:
it_sector = (
    sector_va.query("country_code == 'IT'")
    .groupby(["sector_label", "size_label"], observed=True, as_index=False)["value"]
    .sum()
)
it_sector["size_label"] = pd.Categorical(it_sector["size_label"], SIZE_ORDER_OFFICIAL, ordered=True)
it_sector_share = add_share(it_sector, ["sector_label"])

fig = px.bar(
    it_sector_share.sort_values(["sector_label", "size_label"]),
    x="sector_label",
    y="share_pct",
    color="size_label",
    category_orders={"size_label": SIZE_ORDER_OFFICIAL},
    labels={"sector_label": "Settore", "share_pct": "% valore aggiunto", "size_label": "Classe"},
    title=f"Italia: quota di valore aggiunto per settore e classe - {latest_year}",
)
fig.update_layout(xaxis_tickangle=-35)
add_source_note(fig, "Eurostat Structural Business Statistics")
fig

## Peso delle imprese 250+ nei settori

In [ ]:
sector_share = add_share(
    sector_va.groupby(
        ["country_label", "sector_label", "size_label"],
        observed=True,
        as_index=False,
    )["value"].sum(),
    ["country_label", "sector_label"],
)
large_matrix = sector_share.query("size_label == '250+'").pivot_table(
    index="country_label",
    columns="sector_label",
    values="share_pct",
    aggfunc="sum",
    observed=True,
).round(1)

fig = px.imshow(
    large_matrix,
    aspect="auto",
    color_continuous_scale="Viridis",
    labels={"x": "Settore", "y": "Paese", "color": "% VA 250+"},
    title=f"Quota di valore aggiunto delle imprese 250+ per settore - {latest_year}",
)
add_source_note(fig, "Eurostat Structural Business Statistics")
fig

## Italia: serie per settore

In [ ]:
it_series = (
    sbs.query("metric_code == 'AV_MEUR' and country_code == 'IT'")
    .dropna(subset=["value"])
    .groupby(["year", "sector_label"], observed=True, as_index=False)["value"]
    .sum()
)

fig = px.line(
    it_series,
    x="year",
    y="value",
    color="sector_label",
    markers=True,
    labels={"year": "Anno", "value": "Milioni di euro", "sector_label": "Settore"},
    title="Italia: valore aggiunto per settore nel perimetro SBS",
)
add_source_note(fig, "Eurostat Structural Business Statistics")
fig